> **CeNN memory study:** [Laya V9 strict ternary −1/0/+1 + concat GPU-memory Colab](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V9_Ternary_CeNN_Concat_Memory_Ablation_Colab.ipynb) separately tests 2-bit packed CeNN weights and whether `torch.cat` increases or decreases peak GPU allocation. V9 needs no compiler build or large model download.

# Laya V8 — **Fast, precompiled INT4 CUDA on Tesla T4** (no Triton compiler build)

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V8_T4_Precompiled_BNB_Fast_INT4_Colab.ipynb)

## V7 installation fix

The V7 notebook attempted `pip install git+.../triton-turing`, building a huge custom compiler. **V8 removes the entire fork, ninja/CMake source build, MSLK and TorchAO dependencies.** Instead it uses the **prebuilt bitsandbytes 0.50.2 wheel**, which includes native CUDA fused 4-bit dequantization/GEMM dispatch for **SM75/Tesla T4** (introduced in 0.50.0).

This is honest **W4A16**, not INT4×INT4 arithmetic: weights are actually packed NF4/FP4, activations stay FP16. On supported shapes, bitsandbytes dispatches a fused CUDA GEMM; it can intentionally fall back to FP16 dequantization/GEMM on large M. *Therefore faster end-to-end inference is possible but not guaranteed.*

The experiment probes `mlp.Wi`, `mlp.Wo`, `attn.Wqkv`, `attn.Wo` at M=128/512/2048 for **both NF4 and FP4**. Only shape classes that beat FP16 at both M=512 and M=2048 and pass a robust paired p10 gate are considered. It then tests actual decision agreement on case-disjoint training splits before any module is retained. Final sealed audit and full end-to-end benchmarks establish results.

**Fast no-win exit:** When all precompiled kernels are slower, the notebook skips expensive dataset/model-selection work and still saves a diagnostic ZIP, clearly reporting **0 accepted INT4 projections**.

**Starting weights:** `STARTING_REPO` defaults to the original Laya 1.2 standalone FP16 checkpoint. To benchmark a *V5-derived* FP16 model, change it to the private **standalone V5 FP16 Hugging Face repo**, not the compressed delta-only repository. There is no automatic Hugging Face upload or alteration of your source model.


In [1]:
#@title 1. Fast binary-only installation; DO NOT build or replace Triton
%pip -q install --only-binary=:all: --no-deps "bitsandbytes==0.50.2"
# Keep the Colab-provided torch, CUDA, triton and huggingface_hub untouched.
# Do NOT run the old V7 Turing-fork installation cell.
import sys,importlib.util,subprocess,importlib.metadata
missing=[]
for name,package in [
    ("transformers","transformers==4.57.6"),
    ("datasets","datasets>=2.20,<5"),
    ("safetensors","safetensors>=0.4"),
    ("huggingface_hub","huggingface_hub>=0.35"),
    ("pandas","pandas"),("numpy","numpy")]:
    if importlib.util.find_spec(name) is None:
        missing.append(package)
if missing:
    print("Installing only missing ordinary Python packages:",missing)
    subprocess.run([sys.executable,"-m","pip","install","-q",
                    "--only-binary=:all:",*missing],check=True)
import torch,bitsandbytes as bnb
print("Torch",torch.__version__,"CUDA",torch.version.cuda,
      "bnb",bnb.__version__)
assert bnb.__version__.startswith("0.50."), (
    "Wrong bitsandbytes import; Runtime > Restart session and rerun.")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 18.7 MB/s eta 0:00:00
Torch 2.11.0+cu130 CUDA 13.0 bnb 0.50.2


In [2]:
#@title 2. Experiment configuration and model revision pin
import os,sys,json,gc,copy,random,math,time,zipfile,importlib.util
from collections import Counter
from pathlib import Path
from urllib.request import urlopen
import numpy as np,pandas as pd,torch
from torch import nn
from torch.nn import functional as F
from huggingface_hub import snapshot_download,HfApi
from IPython.display import display
STARTING_REPO="vtava/Laya-1.2-speedup-standalone-fp16" #@param {type:"string"}
QUANT_TYPES="nf4,fp4" #@param {type:"string"}
MIN_OPERATOR_SPEEDUP=1.04 #@param {type:"number"}
MIN_MODEL_SPEEDUP=1.03 #@param {type:"number"}
# Fast default. Increase trials to 40 only if microbenchmarks show gains.
MAX_MODULE_TRIALS=16 #@param {type:"integer"}
MAX_ACCEPTED_MODULES=8 #@param {type:"integer"}
OP_REPEATS=8 #@param {type:"integer"}
MODEL_REPEATS=10 #@param {type:"integer"}
TRAIN_CASES=1200 #@param {type:"integer"}
DEV_DECISIONS=256 #@param {type:"integer"}
GUARD_DECISIONS=128 #@param {type:"integer"}
AUDIT_DECISIONS=384 #@param {type:"integer"}
MAX_NEW_DEV_ERRORS=0 #@param {type:"integer"}
MAX_NEW_GUARD_ERRORS=0 #@param {type:"integer"}
MAX_MEAN_JS=0.0015 #@param {type:"number"}
SEED=2063
OUT=Path("/content/Laya_V8_Precompiled_Int4")
OUT.mkdir(parents=True,exist_ok=True)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
assert torch.cuda.is_available(),"Select a T4/L4/A100 CUDA runtime"
GPU=torch.cuda.get_device_name(0);SM=torch.cuda.get_device_capability(0)
assert SM>=(7,5),"Need CUDA SM75 or newer"
assert MIN_OPERATOR_SPEEDUP>=1 and MODEL_REPEATS>=4 and OP_REPEATS>=4
types=[t.strip() for t in QUANT_TYPES.split(",") if t.strip()]
assert types and set(types).issubset({"nf4","fp4"})
try:
    from google.colab import userdata
    HF_TOKEN=userdata.get("HF_TOKEN")
except Exception:HF_TOKEN=os.environ.get("HF_TOKEN")
SOURCE_SHA=HfApi(token=HF_TOKEN).model_info(STARTING_REPO).sha
ROOT=Path(snapshot_download(repo_id=STARTING_REPO,
                            revision=SOURCE_SHA,token=HF_TOKEN))
meta=json.loads((ROOT/"config.json").read_text())
assert meta.get("format")=="laya-1p2-full-standalone-v1"
assert meta.get("dtype_on_disk")=="float16"
print("GPU",GPU,"SM",SM,"source",STARTING_REPO,SOURCE_SHA)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

GPU Tesla T4 SM (7, 5) source vtava/Laya-1.2-speedup-standalone-fp16 2891f7312d8e7c65b43a6e04c29cc226b06b4fac


In [3]:
#@title 3. Load exact LayaStandalone FP16 and pinned fast binary INT4 helper
def local_import(path,name):
    spec=importlib.util.spec_from_file_location(name,str(path))
    assert spec and spec.loader
    mod=importlib.util.module_from_spec(spec)
    sys.modules[name]=mod;spec.loader.exec_module(mod)
    return mod
source=local_import(ROOT/"modeling_laya_standalone.py","laya_bnb_v8_source")
teacher,tokenizer=source.load_model(ROOT,device="cpu",
                                    local_files_only=True,compile_head=False)
teacher=teacher.to("cuda",dtype=torch.float16).eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
try:
    if hasattr(teacher.encoder,"set_attn_implementation"):
        teacher.encoder.set_attn_implementation("sdpa")
    else:teacher.encoder.config._attn_implementation="sdpa"
except Exception as exc:print("SDPA configuration advisory:",str(exc)[:180])
torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
assert len(teacher.encoder.layers)==28
V8_CODE_SHA="448dde37aed3087801cf28dbba7d40bdd56cbed6"
local=Path("/content/laya_prebuilt_bnb_int4_v8.py")
with urlopen(
    "https://raw.githubusercontent.com/vtavakkoli/TinyCeNN-LM/"+
    V8_CODE_SHA+"/src/tinycenn_lm/laya_prebuilt_bnb_int4_v8.py",
    timeout=90) as res:local.write_bytes(res.read())
assert local.stat().st_size>4000
native=local_import(local,"laya_prebuilt_bnb_int4_v8")
hardware=native.backend_information()
print(json.dumps(hardware,indent=2))


{
  "gpu": "Tesla T4",
  "sm": [
    7,
    5
  ],
  "bitsandbytes_version": "0.50.2",
  "torch_version": "2.11.0+cu130",
  "activation_dtype": "fp16",
  "packed_weight_type": "NF4/FP4",
  "backend": "prebuilt bitsandbytes CUDA 4bit fused GEMM dispatcher",
  "native_int4xint4": false,
  "note": "W4A16 packed inference; selected fused/fallback kernel is shape-dependent."
}


In [4]:
#@title 4. Short real-CUDA 4-bit microbenchmarks: NF4/FP4, M=128/512/2048
PROJECTIONS=("mlp.Wi","mlp.Wo","attn.Wqkv","attn.Wo")
MICRO=[];WINNERS={}
for suffix in PROJECTIONS:
    source_linear=teacher.get_submodule("encoder.layers.0."+suffix)
    for kind in types:
        row={"projection":suffix,"quant_type":kind,"qualified":False}
        try:
            quant=native.as_bnb4(source_linear,quant_type=kind,nested=False)
            assert quant.weight.dtype==torch.uint8
            # Test 3D B*T encoder inputs (not only flattened 2D GEMM)
            x=torch.randn(4,128,source_linear.in_features,
                          device="cuda",dtype=torch.float16)
            y=quant(x)
            assert y.shape==(4,128,source_linear.out_features)
            assert bool(torch.isfinite(y).all())
            timings=native.operator_benchmark(
                source_linear,quant,m_values=(128,512,2048),
                repeats=OP_REPEATS)
            row.update(performance=timings,
                       qualified=native.speed_gate(
                           timings,min_speed=MIN_OPERATOR_SPEEDUP,
                           target_m=(512,2048)))
            gain=min(t["speedup"] for t in timings if t["m"] in (512,2048))
            row["worst_target_speedup"]=gain
            if row["qualified"]:
                if suffix not in WINNERS or gain>WINNERS[suffix]["worst_target_speedup"]:
                    WINNERS[suffix]=row
            print(suffix,kind,"M128/512/2048 speedups:",
                  [round(t["speedup"],3) for t in timings],
                  "qualified",row["qualified"])
        except Exception as exc:
            row["error"]=repr(exc)[:600]
            print("CUDA W4A16 kernel failed:",suffix,kind,repr(exc)[:210])
        MICRO.append(row)
        if "quant" in locals():del quant
        gc.collect();torch.cuda.empty_cache()
(OUT/"operator_microbench_v8.json").write_text(json.dumps(MICRO,indent=2))
(OUT/"fast_kernel_choices.json").write_text(json.dumps(WINNERS,indent=2))
print("Quantization-accelerated shape classes:",list(WINNERS))
SKIP_MODEL_SEARCH=not bool(WINNERS)
if SKIP_MODEL_SEARCH:
    print("EARLY-EXIT: Prebuilt INT4 has no consistent M512/M2048 advantage "
          "on this GPU. No long dataset scan or fake speedup will run.")


mlp.Wi nf4 M128/512/2048 speedups: [0.524, 0.693, 0.87] qualified False
mlp.Wi fp4 M128/512/2048 speedups: [0.542, 0.704, 0.888] qualified False
mlp.Wo nf4 M128/512/2048 speedups: [0.443, 0.574, 0.822] qualified False
mlp.Wo fp4 M128/512/2048 speedups: [0.478, 0.624, 0.832] qualified False
attn.Wqkv nf4 M128/512/2048 speedups: [0.483, 0.622, 0.838] qualified False
attn.Wqkv fp4 M128/512/2048 speedups: [0.495, 0.661, 0.847] qualified False
attn.Wo nf4 M128/512/2048 speedups: [0.403, 0.51, 0.589] qualified False
attn.Wo fp4 M128/512/2048 speedups: [0.423, 0.514, 0.72] qualified False
Quantization-accelerated shape classes: []
EARLY-EXIT: Prebuilt INT4 has no consistent M512/M2048 advantage on this GPU. No long dataset scan or fake speedup will run.


In [5]:
#@title 5. Only if fast kernels exist: separate development, guard, and sealed audit
SEL_ITEMS=[];GUARD_ITEMS=[];AUDIT_ITEMS=[]
SEL=[];GUARD=[];AUDIT=[]
REF_SEL=[];REF_GUARD=[];REF_AUDIT=[]
def to_obj(val):
    if isinstance(val,str):
        try:return json.loads(val)
        except Exception:return val
    return val
def prepare_items(rows,limit):
    cases=[]
    for row in rows:
        state=to_obj(row.get("state"))
        questions=to_obj(row.get("questions"))
        if not isinstance(questions,dict):continue
        group=[]
        for raw in questions.values():
            try:
                question=source.normalize_question(to_obj(raw))
                ids,markers=source.build_sequence(tokenizer,state,question,
                    max_len=512,head_max_len=192,truncate_left=isinstance(state,list))
                if len(markers)<2:continue
                group.append(dict(ids=ids,markers=markers,
                                  qtype=source.QTYPES[question["t"]]))
            except (ValueError,TypeError,KeyError,IndexError):continue
        if group:cases.append(group)
    items=[]
    for depth in range(max(map(len,cases),default=0)):
        for group in cases:
            if depth<len(group):
                items.append(group[depth])
                if len(items)>=limit:return items
    return items
def as_batches(items,size=4,pad_to=None):
    result=[]
    for i in range(0,len(items),size):
        b=source.collate_items(items[i:i+size],int(tokenizer.pad_token_id))
        if pad_to is not None:
            missing=pad_to-b["input_ids"].shape[1]
            if missing<0:raise ValueError("Sequence exceeds benchmark length")
            b["input_ids"]=F.pad(b["input_ids"],(0,missing),
                                 value=int(tokenizer.pad_token_id))
            b["attention_mask"]=F.pad(b["attention_mask"],(0,missing))
        result.append({k:v.to("cuda") for k,v in b.items()})
    return result
@torch.inference_mode()
def infer(model,b):
    with torch.autocast("cuda",dtype=torch.float16):
        result=model(**b)
    return result[0] if isinstance(result,(tuple,list)) else result
@torch.inference_mode()
def compare(model,batches,ref):
    errors=0;n=0;js_total=0.;mask=[]
    for batch,baseline in zip(batches,ref):
        logits=infer(model,batch).float().cpu()
        target=baseline.float().cpu()
        valid=batch["marker_mask"].bool().cpu()
        logits=logits.masked_fill(~valid,-1.e4)
        target=target.masked_fill(~valid,-1.e4)
        p=logits.softmax(-1).clamp_min(1e-9)
        q=target.softmax(-1).clamp_min(1e-9)
        mid=(p+q)*.5
        js=.5*(p*(p/mid).log()).sum(-1)+.5*(q*(q/mid).log()).sum(-1)
        wrong=(p.argmax(-1)!=q.argmax(-1))
        mask.extend(wrong.tolist())
        errors+=int(wrong.sum());n+=len(wrong)
        js_total+=float(js.sum())
    return dict(n=n,errors=errors,agreement=1-errors/n,
                mean_js=js_total/n,error_mask=mask)
if not SKIP_MODEL_SEARCH:
    from datasets import load_dataset
    rows=list(load_dataset("LocalLLaMA/typed-decisions","all",
                           split=f"train[:{TRAIN_CASES}]"))
    random.Random(SEED).shuffle(rows)
    cuts=[0,int(len(rows)*.10),int(len(rows)*.40),
          int(len(rows)*.55),len(rows)]
    SEL_ITEMS=prepare_items(rows[cuts[1]:cuts[2]],DEV_DECISIONS)
    GUARD_ITEMS=prepare_items(rows[cuts[2]:cuts[3]],GUARD_DECISIONS)
    AUDIT_ITEMS=prepare_items(rows[cuts[3]:],AUDIT_DECISIONS)
    assert [len(SEL_ITEMS),len(GUARD_ITEMS),len(AUDIT_ITEMS)]==[
        DEV_DECISIONS,GUARD_DECISIONS,AUDIT_DECISIONS], (
        "Insufficient independent source-case questions")
    SEL,GUARD,AUDIT=[as_batches(items) for items in
                     (SEL_ITEMS,GUARD_ITEMS,AUDIT_ITEMS)]
    REF_SEL=[infer(teacher,b).float().cpu() for b in SEL]
    REF_GUARD=[infer(teacher,b).float().cpu() for b in GUARD]
    # Do not evaluate AUDIT until after all precision selection is frozen.
    print("Loaded disjoint SELECT/GUARD; AUDIT kept sealed")
else:print("Skipped dataset download and forward passes: no fast CUDA kernel.")


Skipped dataset download and forward passes: no fast CUDA kernel.


In [6]:
#@title 6. Select true packed NF4/FP4 projections with no new development mistakes
STUDENT=None;CHOSEN=[];TRIALS=[]
BASE_SEL=BASE_GUARD=None
CURRENT_SEL=CURRENT_GUARD=None
if not SKIP_MODEL_SEARCH:
    STUDENT=copy.deepcopy(teacher).eval().requires_grad_(False)
    BASE_SEL=compare(STUDENT,SEL,REF_SEL)
    BASE_GUARD=compare(STUDENT,GUARD,REF_GUARD)
    CURRENT_SEL,CURRENT_GUARD=BASE_SEL,BASE_GUARD
    # Filter before slicing! Larger MLP.Wi layers would otherwise consume
    # every trial slot even if only (for example) mlp.Wo won the microbench.
    names=[n for n in native.locate_candidates(teacher)
           if n.split(".",3)[-1] in WINNERS]
    print("Eligible model projection sites:",len(names),
          "| trial budget:",min(len(names),MAX_MODULE_TRIALS))
    for index,name in enumerate(names[:MAX_MODULE_TRIALS],1):
        if len(CHOSEN)>=MAX_ACCEPTED_MODULES:break
        suffix=name.split(".",3)[-1]
        backend=WINNERS[suffix]["quant_type"]
        old=STUDENT.get_submodule(name)
        trial=dict(name=name,quant_type=backend,accepted=False)
        try:
            candidate=native.as_bnb4(old,backend,nested=False)
            native.replace_module(STUDENT,name,candidate)
            s=compare(STUDENT,SEL,REF_SEL)
            g=compare(STUDENT,GUARD,REF_GUARD)
            new_s=sum(b and not a for a,b in zip(
                CURRENT_SEL["error_mask"],s["error_mask"]))
            new_g=sum(b and not a for a,b in zip(
                CURRENT_GUARD["error_mask"],g["error_mask"]))
            keep=(new_s<=MAX_NEW_DEV_ERRORS and new_g<=MAX_NEW_GUARD_ERRORS
                  and s["errors"]<=CURRENT_SEL["errors"]
                  and g["errors"]<=CURRENT_GUARD["errors"]
                  and s["mean_js"]<=MAX_MEAN_JS
                  and g["mean_js"]<=MAX_MEAN_JS)
            trial.update(select_errors=s["errors"],guard_errors=g["errors"],
                         new_select_errors=new_s,new_guard_errors=new_g,
                         accepted=bool(keep))
            if keep:
                CURRENT_SEL,CURRENT_GUARD=s,g
                CHOSEN.append(dict(name=name,quant_type=backend))
                print("ACCEPT W4A16",name,backend,
                      "DEV",round(s["agreement"],4),
                      "GUARD",round(g["agreement"],4))
            else:native.replace_module(STUDENT,name,old)
        except Exception as exc:
            native.replace_module(STUDENT,name,old)
            trial["error"]=repr(exc)[:350]
            print("Reject",name,repr(exc)[:160])
        TRIALS.append(trial)
        if index%4==0:
            (OUT/"model_trials_v8.json").write_text(json.dumps(TRIALS,indent=2))
    (OUT/"model_trials_v8.json").write_text(json.dumps(TRIALS,indent=2))
print("Actual packed 4-bit modules retained:",len(CHOSEN))


Actual packed 4-bit modules retained: 0


In [7]:
#@title 7. Sealed audit and full-model paired speed, only for actual INT4
AUDIT_RESULT=None;DEV_RESULT=None;GUARD_RESULT=None;BENCH=[]
AUDIT_PASS=False;SPEED_PASS=False;VERIFIED=False
if CHOSEN:
    DEV_RESULT=compare(STUDENT,SEL,REF_SEL)
    GUARD_RESULT=compare(STUDENT,GUARD,REF_GUARD)
    # Independent sealed audit is evaluated once only, after selection.
    REF_AUDIT=[infer(teacher,b).float().cpu() for b in AUDIT]
    AUDIT_RESULT=compare(STUDENT,AUDIT,REF_AUDIT)
    AUDIT_PASS=(AUDIT_RESULT["agreement"]>=0.99
                and AUDIT_RESULT["mean_js"]<=MAX_MEAN_JS)
    for B in (1,4):
        group=SEL_ITEMS[:B]
        minlen=max(len(x["ids"]) for x in group)
        for T in (128,256,512):
            if minlen>T:continue
            batch=as_batches(group,size=B,pad_to=T)[0]
            record=native.paired_cuda_ms(
                lambda:infer(teacher,batch),
                lambda:infer(STUDENT,batch),repeats=MODEL_REPEATS)
            BENCH.append(dict(batch=B,tokens=T,**record))
            print("E2E B",B,"T",T,"speed",round(record["speedup"],3),
                  "p10",round(record["p10_speedup"],3))
    table=pd.DataFrame(BENCH)
    display(table)
    table.to_csv(OUT/"end_to_end_speed_v8.csv",index=False)
    targets=table[(table.tokens==512)&table.batch.isin([1,4])]
    SPEED_PASS=(len(targets)==2
                and bool((targets.speedup>=MIN_MODEL_SPEEDUP).all())
                and bool((targets.p10_speedup>=1.0).all()))
    VERIFIED=bool(CHOSEN) and AUDIT_PASS and SPEED_PASS
    print("Independent AUDIT",{
        k:v for k,v in AUDIT_RESULT.items() if k!="error_mask"})
    print("W4A16 model speed AND quality validated:",VERIFIED)
else:
    print("No real INT4 projections installed; no quantized-model AUDIT is "
          "reported. No fake FP16-vs-FP16 speedup.")


No real INT4 projections installed; no quantized-model AUDIT is reported. No fake FP16-vs-FP16 speedup.


In [8]:
#@title 8. Reproducible result ZIP and qualification summary; no source HF changes
def public(score):
    return {k:v for k,v in (score or {}).items() if k!="error_mask"}
result=dict(
    experiment="Laya V8 precompiled bitsandbytes 0.50.2 packed W4A16",
    source_repo=STARTING_REPO,source_revision=SOURCE_SHA,
    torch_version=str(torch.__version__),cuda_version=torch.version.cuda,
    bitsandbytes_version=hardware["bitsandbytes_version"],
    gpu=GPU,compute_capability=list(SM),
    candidate_shape_classes=list(WINNERS),
    int4_modules=CHOSEN,selected_count=len(CHOSEN),
    projection_total=112,weight_bits=4,activation_dtype="float16",
    uses_fused_or_fallback_gemm_dispatch=True,
    native_int4xint4=False,
    validation="no_int4_kernel_speedup" if SKIP_MODEL_SEARCH
               else ("no_accuracy_qualified_int4" if not CHOSEN
                     else ("validated" if VERIFIED else "not_validated")),
    dev=public(DEV_RESULT),guard=public(GUARD_RESULT),
    independent_audit=public(AUDIT_RESULT),
    audit_pass=bool(AUDIT_PASS),
    e2e_speed_pass=bool(SPEED_PASS),verified=bool(VERIFIED),
    model_benchmarks=BENCH,
    original_baseline_fp16_bytes=native.gpu_checkpoint_bytes(teacher),
    optimized_state_bytes=(native.gpu_checkpoint_bytes(STUDENT)
                           if STUDENT is not None else None),
    note=(
      "Prebuilt CUDA 4bit weights are real; bnb may fallback to "
      "dequantize+FP16 GEMM for some large M. No speedup claim "
      "without an independent audit and measured end-to-end win.")
)
(OUT/"summary_prebuilt_int4_v8.json").write_text(json.dumps(result,indent=2))
(OUT/"selection_manifest_v8.json").write_text(json.dumps({
    "base":STARTING_REPO,"sha":SOURCE_SHA,"bnb_version":bnb.__version__,
    "dtype":"NF4/FP4 packed","modules":CHOSEN,
    "validated":bool(VERIFIED),
    "reconstruction_requires_original_fp16_base":True},indent=2))
archive=OUT/"Laya_V8_T4_Precompiled_INT4_Results.zip"
with zipfile.ZipFile(archive,"w",zipfile.ZIP_DEFLATED) as z:
    for file in OUT.iterdir():
        if file.is_file() and file!=archive:z.write(file,file.name)
print("Result ZIP:",archive,"status:",result["validation"])
try:
    from google.colab import files
    files.download(str(archive))
except ImportError:pass


Result ZIP: /content/Laya_V8_Precompiled_Int4/Laya_V8_T4_Precompiled_INT4_Results.zip status: no_int4_kernel_speedup


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## What V8 proves

The prebuilt bitsandbytes CUDA wheel supports T4 NF4/FP4 quantization without compiling any third-party compiler. It operates on packed 4-bit weights and FP16 activations; the fused GEMM dispatch can fall back depending on matrix shape. This is **not** an INT4×INT4 Tensor Core kernel.

**Positive:** at least one INT4 layer is installed; sealed audit passes; and end-to-end B1×T512 and B4×T512 both beat FP16 with a robust timing margin.

**Negative:** no compatible faster shape class, no accepted layers, failed audit, or no end-to-end gain. The notebook writes a diagnostic ZIP in any of those cases and does not mislabel an unchanged FP16 model as successful INT4.

To accelerate your V5 checkpoint rather than the original Laya, export V5 as **standalone FP16** using the last cell of the V5 Colab and point `STARTING_REPO` to that model.

No model is uploaded automatically. The selection manifest records a reproducible recipe, not a validated standalone bnb serialized model.
